# Stage 7 — Package and publish

**Goal:** turn a training checkpoint into an artifact a stranger can load.

### A checkpoint and a model repo are different things

| | checkpoint (`.pt`) | model repo |
|---|---|---|
| Purpose | resume training | be loaded by someone else |
| Contains | weights, optimizer moments, RNG state, step | weights, architecture, tokenizer, docs |
| Meaningful to | the code that wrote it | anyone |
| Size here | ~190 MB (3× the model — Adam's two moments) | ~63 MB |

### What goes in the repo, and why each file matters

| file | why |
|---|---|
| `model.safetensors` | weights. Not `.bin`: loading a pickle **executes arbitrary code**, and a model repo is by definition untrusted input |
| `config.json` | architecture — what tells transformers, and later llama.cpp, that this is a Llama model |
| `generation_config.json` | default sampling params, so `generate()` behaves sensibly without the caller knowing details |
| `tokenizer.model` | the SentencePiece vocabulary. **Stage 8 depends on this exact filename** |
| `tokenizer_config.json` | special tokens and the chat template |
| `README.md` | the model card |

### On model cards

A model card is not decoration. It is the only place someone can learn what the
model was trained on, what it's for, and where it fails. Ours states bluntly that
the model does one thing and produces confident nonsense outside it — because
that is true, and a card that oversells is worse than no card.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Set this to YOUR GitHub repo once; every notebook uses the same cell.
REPO_URL = "https://github.com/pythonstudentiam/e2e_llm_demo.git"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/e2e_llm_demo")
WORK = Path("/content/work")          # scratch: data + checkpoints (ephemeral!)
WORK.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

# Colab ships torch; these are the rest. -q to keep the log readable.
%pip install -q sentencepiece "datasets>=3.0" "transformers>=4.45" "huggingface_hub>=0.30"

# HF token from the Colab Secrets panel (key icon, left sidebar). Name it
# HF_TOKEN and enable Notebook access -- the grant is PER NOTEBOOK, so every
# notebook asks separately. Never paste a token into a cell.
#
# login() rather than just setting the env var: it writes the token where every
# huggingface_hub call looks, including ones that ignore the environment.
try:
    from google.colab import userdata
    from huggingface_hub import login

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF_TOKEN loaded from Colab Secrets, authenticated")
except Exception as e:
    print("=" * 72)
    print(f"  HF_TOKEN IS NOT AVAILABLE  ({type(e).__name__}: {e})")
    print()
    print("  Every Hub call in this notebook will fail with 401 Unauthorized.")
    print("  Fix: click the key icon in the left sidebar, turn on Notebook")
    print("       access for HF_TOKEN, then RE-RUN THIS CELL before continuing.")
    print("=" * 72)

import torch
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# This stage only does forward passes, so it works on CPU -- just slower.
# Free-tier GPU quota runs out; that should slow you down, not block you.
import torch

if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)} | compute capability "
          f"{'.'.join(map(str, torch.cuda.get_device_capability(0)))}")
else:
    print("No GPU -- running on CPU. Everything here works, roughly 5-10x slower.")
    print("(Runtime > Change runtime type > T4 GPU if you have quota available.)")

In [ ]:
from tinyllm import config
from tinyllm.config import (
    model_cfg, train_cfg, data_cfg, tok_cfg, sft_cfg, gen_cfg, quant_cfg, serve_cfg, hub,
)

print(config.summary())

## 7.1 — Load the instruction-tuned model

In [ ]:
from pathlib import Path
import torch, json, numpy as np
from huggingface_hub import hf_hub_download
from tinyllm.tokenizer import load_sp
from tinyllm.train import build_model, pull_checkpoint
from tinyllm.data import load_tokens

tok_dir = WORK / "tokenizer"
sp_path = tok_dir / "tokenizer.model"
if not sp_path.exists():
    tok_dir.mkdir(parents=True, exist_ok=True)
    got = hf_hub_download(repo_id=hub.ckpt_repo, filename="tokenizer/tokenizer.model")
    sp_path.write_bytes(Path(got).read_bytes())
sp = load_sp(sp_path)

# Prefer the instruction-tuned model. Fall back to the base model so this stage
# is not blocked by notebook 06 -- useful when GPU quota runs out, since you can
# still package, convert and serve a working (text-completion) model today and
# re-run this notebook after SFT.
IS_CHAT_MODEL = True

ckpt = WORK / "checkpoints/sft/sft_final.pt"
if not ckpt.exists():
    ckpt = pull_checkpoint(filename="sft_final.pt", local_dir=WORK / "checkpoints/sft")

if not (ckpt and Path(ckpt).exists()):
    print("=" * 72)
    print("  No SFT checkpoint found -- falling back to the BASE model.")
    print("  You will publish a text-completion model, not a chat model.")
    print("  Re-run this notebook after notebook 06 to publish the chat version.")
    print("=" * 72)
    IS_CHAT_MODEL = False
    ckpt = WORK / "checkpoints/latest.pt"
    if not ckpt.exists():
        ckpt = pull_checkpoint(local_dir=WORK / "checkpoints")

assert ckpt and Path(ckpt).exists(), "No checkpoint at all -- run notebook 04 first."

model = build_model(model_cfg)          # picks CUDA if available, else CPU
model.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=False)["model"])
model.eval()
kind = "instruction-tuned" if IS_CHAT_MODEL else "base"
print(f"loaded {kind} model: {sum(p.numel() for p in model.parameters()):,} params")

## 7.2 — Export

`chat_model=True` sets EOS to `<|im_end|>`. This matters at serving time:
`llama-server` stops generating at the GGUF's EOS id, and for a ChatML model that
has to be the **turn terminator**, not the end-of-document token. Get this wrong
and the server runs on past the end of the reply until it hits the token limit.

In [ ]:
from tinyllm.export import export_model

eval_report = None
p = WORK / "reports/base_eval.json"
if p.exists():
    eval_report = json.loads(p.read_text())

export_dir = WORK / "export/tinyllm"
export_model(model, sp_path, export_dir, chat_model=True, eval_report=eval_report)

print("exported repo:")
for f in sorted(export_dir.iterdir()):
    print(f"  {f.name:<30} {f.stat().st_size:>10,} B")

## 7.3 — The stage 7 gate: reload it from disk

Round-tripping through `from_pretrained` is what catches a dtype change, a
dropped tied weight, or a config field that didn't serialize. If this passes, the
repo works for anyone who downloads it.

In [ ]:
from tinyllm.export import verify_export

result = verify_export(export_dir, reference_model=model, device="cpu")
for k, v in result.items():
    if k != "files":
        print(f"  {k:<20} {v}")
print(f"  files: {', '.join(result['files'])}")

In [ ]:
# And confirm the chat template survives the round trip -- this is the string
# that has to reach llama-server intact.
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(str(export_dir))
messages = [{"role": "user", "content": "Write a story about a lost puppy."}]
rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("chat template renders to:\n")
print(repr(rendered))
assert "<|im_start|>assistant" in rendered, "template is not producing a generation prompt"
print("\nEOS token:", tok.eos_token, "->", tok.eos_token_id)

## 7.4 — Push to the Hub

In [ ]:
from tinyllm.export import push_to_hub

url = push_to_hub(export_dir, private=False, commit_message="tinyllm: instruction-tuned")
print(f"live at {url}")

In [ ]:
# Prove it: load the model back from the Hub in a fresh process-like context,
# by repo id rather than local path. This is what a stranger's code does.
from transformers import AutoModelForCausalLM, AutoTokenizer

remote_tok = AutoTokenizer.from_pretrained(hub.model_repo)
remote_model = AutoModelForCausalLM.from_pretrained(hub.model_repo)
print(f"downloaded {sum(p.numel() for p in remote_model.parameters()):,} params from {hub.model_repo}")

prompt = remote_tok.apply_chat_template(
    [{"role": "user", "content": "Write a story about a brave little boat."}],
    tokenize=False, add_generation_prompt=True)
ids = remote_tok(prompt, return_tensors="pt")
out = remote_model.generate(**ids, max_new_tokens=180, do_sample=True, temperature=0.8)
print("\n" + remote_tok.decode(out[0], skip_special_tokens=True))

del remote_model

## Stage 7 gate

- [x] Exported as safetensors with config, tokenizer, and generation config
- [x] Reload from disk produces identical logits
- [x] Chat template renders a generation prompt correctly
- [x] Model card written, including honest limitations
- [x] Loads from the Hub by repo id

**Next:** `08_gguf.ipynb`